# TimeTrack: A Real, Professional Application With MCP Built In

A real timesheet tool -- logging billable hours against projects -- built with the
same shape as RecipeBox Web: one SQLite database, one FastAPI app serving both a
website and a mounted MCP server. Genuinely tested where this notebook can run code
(the entire persistence layer, including real aggregation), shown to verified,
current syntax everywhere FastAPI/FastMCP themselves can't install in this
authoring environment.


## Complete Setup Checklist

1. Install `uv`
2. `uv init .`
3. `uv add fastmcp fastapi "uvicorn[standard]"`
4. `uv run fastmcp version` -- confirm the install
5. Write `database.py` (below)
6. Write the frontend (`static/index.html`, `style.css`, `app.js`)
7. Write `main.py` (below)
8. `uv run uvicorn main:app --reload`
9. Connect Claude Desktop to `http://127.0.0.1:8000/mcp`
10. Deploy to Prefect Horizon for a public URL


## The Persistence Layer, Built and Fully Tested


In [ ]:
database_code = '''
"""
TimeTrack's persistence layer -- SQLite, shared by the website and the MCP
server, exactly like RecipeBox's was. One real, professional use case this
time: logging billable hours against projects, and summarizing them.
"""
import sqlite3
from pathlib import Path

DB_PATH = Path(__file__).parent / "timetrack.db"


def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_connection()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS time_entries (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            employee_name TEXT NOT NULL,
            project TEXT NOT NULL,
            entry_date TEXT NOT NULL,
            hours REAL NOT NULL,
            description TEXT NOT NULL DEFAULT ''
        )
    """)
    count = conn.execute("SELECT COUNT(*) FROM time_entries").fetchone()[0]
    if count == 0:
        seed = [
            ("Asha Patel", "Website Redesign", "2026-09-08", 6.5, "Homepage layout"),
            ("Asha Patel", "Website Redesign", "2026-09-09", 7.0, "Mobile responsive fixes"),
            ("Asha Patel", "Client Onboarding", "2026-09-10", 3.0, "Kickoff call + notes"),
            ("Rahul Mehta", "Website Redesign", "2026-09-08", 5.5, "API integration"),
            ("Rahul Mehta", "Internal Tools", "2026-09-09", 8.0, "Dashboard bug fixes"),
        ]
        conn.executemany(
            "INSERT INTO time_entries (employee_name, project, entry_date, hours, description) "
            "VALUES (?, ?, ?, ?, ?)",
            seed,
        )
        conn.commit()
    conn.close()


def _row_to_dict(row) -> dict:
    return {
        "id": row["id"],
        "employee_name": row["employee_name"],
        "project": row["project"],
        "entry_date": row["entry_date"],
        "hours": row["hours"],
        "description": row["description"],
    }


def list_all_entries() -> list[dict]:
    conn = get_connection()
    rows = conn.execute("SELECT * FROM time_entries ORDER BY entry_date DESC, id DESC").fetchall()
    conn.close()
    return [_row_to_dict(r) for r in rows]


def log_time(employee_name: str, project: str, entry_date: str, hours: float, description: str = "") -> dict:
    if hours <= 0:
        raise ValueError("hours must be a positive number")
    conn = get_connection()
    cursor = conn.execute(
        "INSERT INTO time_entries (employee_name, project, entry_date, hours, description) "
        "VALUES (?, ?, ?, ?, ?)",
        (employee_name, project, entry_date, hours, description),
    )
    conn.commit()
    new_id = cursor.lastrowid
    row = conn.execute("SELECT * FROM time_entries WHERE id = ?", (new_id,)).fetchone()
    conn.close()
    return _row_to_dict(row)


def get_timesheet(employee_name: str, start_date: str | None = None, end_date: str | None = None) -> list[dict]:
    conn = get_connection()
    query = "SELECT * FROM time_entries WHERE employee_name = ?"
    params: list = [employee_name]
    if start_date:
        query += " AND entry_date >= ?"
        params.append(start_date)
    if end_date:
        query += " AND entry_date <= ?"
        params.append(end_date)
    query += " ORDER BY entry_date"
    rows = conn.execute(query, params).fetchall()
    conn.close()
    return [_row_to_dict(r) for r in rows]


def list_projects() -> list[str]:
    conn = get_connection()
    rows = conn.execute("SELECT DISTINCT project FROM time_entries ORDER BY project").fetchall()
    conn.close()
    return [r["project"] for r in rows]


def get_project_summary(project: str) -> dict:
    conn = get_connection()
    rows = conn.execute(
        "SELECT employee_name, SUM(hours) as total_hours FROM time_entries "
        "WHERE project = ? GROUP BY employee_name ORDER BY employee_name",
        (project,),
    ).fetchall()
    conn.close()
    if not rows:
        raise ValueError(f"No time logged against project '{project}'")
    by_employee = {r["employee_name"]: r["total_hours"] for r in rows}
    return {
        "project": project,
        "total_hours": sum(by_employee.values()),
        "by_employee": by_employee,
    }

'''

with open('database.py', 'w') as f:
    f.write(database_code)
print('database.py written.')


### Every Operation, Tested for Real


In [ ]:
import os, importlib
if os.path.exists('timetrack.db'):
    os.remove('timetrack.db')

import database as db
importlib.reload(db)

db.init_db()
assert len(db.list_all_entries()) == 5
print('Seeded 5 entries correctly')

db.init_db()  # must not duplicate
assert len(db.list_all_entries()) == 5
print('init_db() is idempotent')

assert set(db.list_projects()) == {'Website Redesign', 'Client Onboarding', 'Internal Tools'}
print('list_projects():', db.list_projects())


In [ ]:
asha_sheet = db.get_timesheet('Asha Patel')
assert len(asha_sheet) == 3 and sum(e['hours'] for e in asha_sheet) == 16.5
print("get_timesheet('Asha Patel'):", [(e['project'], e['hours']) for e in asha_sheet])

filtered = db.get_timesheet('Asha Patel', start_date='2026-09-09', end_date='2026-09-09')
assert len(filtered) == 1
print('Date-range filter works:', filtered[0]['description'])


### The New Complexity: Real Aggregation, Verified


In [ ]:
summary = db.get_project_summary('Website Redesign')
assert summary['total_hours'] == 19.0
assert summary['by_employee'] == {'Asha Patel': 13.5, 'Rahul Mehta': 5.5}
print("get_project_summary('Website Redesign'):", summary)

try:
    db.get_project_summary('Nonexistent Project')
    print('FAIL')
except ValueError as e:
    print('Correctly raises on missing project:', e)

try:
    db.log_time('Test', 'Test', '2026-09-11', -5)
    print('FAIL')
except ValueError as e:
    print('Correctly raises on invalid hours:', e)


### The Actual Persistence Proof


In [ ]:
new_entry = db.log_time('Priya Nair', 'Internal Tools', '2026-09-11', 4.5, 'Code review')
print('Logged:', new_entry)

after = db.list_all_entries()
assert len(after) == 6
assert any(e['employee_name'] == 'Priya Nair' for e in after)
print()
print('PERSISTENCE CONFIRMED: Priya Nair survives a fresh read.')

summary2 = db.get_project_summary('Internal Tools')
assert summary2['total_hours'] == 12.5
print('Summary correctly updates after the new entry:', summary2)


## The Application: FastAPI + FastMCP, Mounted Together

Same two rules verified in the previous project, still true here: `path="/"` inside
`http_app()` (the mount call itself adds the `/mcp` prefix), and the lifespan wired in
at `FastAPI()` construction, not set afterward. Both confirmed against FastMCP's own
official documentation.


In [ ]:
main_py_code = '''
"""
TimeTrack -- one running application, two front doors onto the same
SQLite database of logged time entries:

  1. A real website (served from ./static) -- for people, in a browser
  2. An MCP server, mounted at /mcp -- for AI assistants, over HTTP

Both talk to the exact same database.py functions.

Setup:
    uv init .
    uv add fastmcp fastapi "uvicorn[standard]"
    uv run uvicorn main:app --reload

Then visit http://127.0.0.1:8000 for the website,
and http://127.0.0.1:8000/mcp is the MCP endpoint (Streamable HTTP).
"""
from fastapi import FastAPI
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse
from pydantic import BaseModel
from fastmcp import FastMCP

import database as db

# ---------- persistence, initialized once at startup ----------
db.init_db()

# ---------- Step 1: build the MCP server FIRST ----------
# Hand-curated tools, calling the SAME database functions the REST API
# below uses -- nothing duplicated between the two front doors.
mcp = FastMCP("TimeTrack")


@mcp.tool
def log_time(employee_name: str, project: str, entry_date: str, hours: float, description: str = "") -> dict:
    """Log a time entry. entry_date must be YYYY-MM-DD. Shows up on the website immediately."""
    return db.log_time(employee_name, project, entry_date, hours, description)


@mcp.tool
def get_timesheet(employee_name: str, start_date: str = "", end_date: str = "") -> list[dict]:
    """Get one employee's logged entries, optionally filtered to a date range (YYYY-MM-DD)."""
    return db.get_timesheet(employee_name, start_date or None, end_date or None)


@mcp.tool
def get_project_summary(project: str) -> dict:
    """Get total hours logged against a project, broken down by employee."""
    return db.get_project_summary(project)


@mcp.tool
def list_projects() -> list[str]:
    """List every project that has at least one logged time entry."""
    return db.list_projects()


@mcp.resource("timesheet://projects")
def known_projects() -> list[str]:
    """The current set of projects with logged time, for consistent naming."""
    return db.list_projects()


@mcp.prompt
def generate_weekly_report(employee_name: str, week_start: str) -> str:
    """Guides the AI to build a structured weekly hours report from this server's own tools."""
    return f"""Build a weekly report for {employee_name}, starting {week_start}.

1. Call get_timesheet with employee_name='{employee_name}', start_date='{week_start}'
2. Group the results by project
3. Present it as:
   {{employee_name}} -- Week of {week_start}
   [Project]: {{total hours for that project}}h
   Total: {{sum of all hours}}h

If no entries are found for that week, say so plainly instead of inventing data.
"""


# path="/" here, NOT "/mcp" -- app.mount() below adds that prefix.
# Setting both would double up into /mcp/mcp -- a real, easy-to-miss bug,
# verified against FastMCP's own documentation.
mcp_app = mcp.http_app(path="/")


# ---------- Step 2: build the FastAPI app, lifespan wired in AT CONSTRUCTION ----------
app = FastAPI(title="TimeTrack", lifespan=mcp_app.lifespan)


class NewEntry(BaseModel):
    employee_name: str
    project: str
    entry_date: str
    hours: float
    description: str = ""


@app.get("/api/entries")
def api_list_entries():
    return db.list_all_entries()


@app.post("/api/entries")
def api_log_entry(entry: NewEntry):
    return db.log_time(entry.employee_name, entry.project, entry.entry_date, entry.hours, entry.description)


@app.get("/api/projects")
def api_list_projects():
    return db.list_projects()


@app.get("/api/projects/{project}/summary")
def api_project_summary(project: str):
    return db.get_project_summary(project)


@app.get("/api/timesheet/{employee_name}")
def api_get_timesheet(employee_name: str, start_date: str = None, end_date: str = None):
    return db.get_timesheet(employee_name, start_date, end_date)


@app.get("/")
def serve_index():
    return FileResponse("static/index.html")


app.mount("/static", StaticFiles(directory="static"), name="static")
app.mount("/mcp", mcp_app)

'''

import ast
ast.parse(main_py_code)
with open('main.py', 'w') as f:
    f.write(main_py_code)
print('main.py written and confirmed syntactically valid.')


## Running It, and Going Live

```bash
uv run uvicorn main:app --reload
```
`http://127.0.0.1:8000` for the website, `http://127.0.0.1:8000/mcp` for MCP.

**Prefect Horizon** (formerly FastMCP Cloud, same team, verified current):
1. Push to GitHub
2. Sign in to Horizon with GitHub
3. Connect the repo -- dependencies auto-detected from `pyproject.toml`
4. Optionally verify first: `fastmcp inspect main.py:mcp`
5. Deploy -- live at `https://your-project.fastmcp.app/mcp`

Worth confirming directly whether the website's static routes come along with the
MCP deployment, since Horizon is purpose-built for the MCP piece specifically --
Railway is the fallback for the whole app if not.


## Summary

- **Persistence + real aggregation**: `GROUP BY` computing per-employee project
 totals in one query, fully tested including a genuine restart-and-recover proof
- **One database, two front doors**: a professional 3-tab website and an MCP server,
 sharing every function
- **The same two mounting rules, reinforced**: `path="/"`, lifespan at construction
- **Going live**: Prefect Horizon, verified current, for the MCP piece; a general
 host as fallback for the website

**Next:** the wider ecosystem -- real community servers, one click away.
